# C10-competition-craft — Session 2: Metric-Driven Iteration

*One class session, roughly 85 minutes. Prerequisites: Session 1 (the
contract, the hidden-test protocol, the apiary harness), C4 (pipelines,
validation splits), and the metric family taught in C1 and practiced
in C4 (accuracy, precision,
recall, F1, macro-F1).*

**This session:** a competition is decided by a *number*, and the first
craft skill is knowing exactly which number.
The apiary task — like the exam's applied problem — is graded by
**macro-F1**, not accuracy, and on a 2:1-imbalanced table the two can
disagree spectacularly.
We rebuild macro-F1 from its confusion-matrix pieces (C1's formulas,
applying C4's trust-but-test habit: the first time you call
`sklearn.metrics.confusion_matrix`, verify its orientation on a tiny
hand-checkable example — rows are true classes, columns are predictions,
both in sorted label order —
now derived into one pipeline-ready computation), install validation as
the *only honest signal* of held-back performance, and then practice
the loop the whole sport runs on: **baseline → error analysis → one
change → re-validate**, with a log, a cap, and a warning about wearing
the validation split out.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()

## 1. The Scoreboard Question

**Motivation.**
C1 showed accuracy actively lying under class imbalance: a screening
rule that never fires can be 99% "accurate".
Our table is 63% `thrives`, so the do-nothing rule — every colony
thrives — banks 0.63 accuracy while telling the apiary *nothing*: the
colonies that need help are exactly the ones it never flags.

A competition that graded accuracy here would reward contestants for
polishing the majority class and quietly abandoning the minority.
So this task (and the exam's applied problem) grades **macro-F1**: the
mean of the per-class F1 scores, where the 37% minority class counts
exactly as much as the 63% majority.
From C1: "macro" *means* every class weighs equally.

Watch both scoreboards judge the do-nothing rule on a validation
split (the pinned carve — seeded, stratified — that this whole unit
reuses):

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y)
print("carve:", X_tr.shape, X_val.shape,
      "| val counts:", dict(zip(*np.unique(y_val, return_counts=True))))

always_thrives = np.full(len(y_val), "thrives")
acc_baseline = (always_thrives == y_val).mean()
f1_baseline = f1_score(y_val, always_thrives, average="macro")
print(f"do-nothing rule: accuracy = {acc_baseline:.4f}"
      f" | macro-F1 = {f1_baseline:.4f}")

Accuracy 0.6333, macro-F1 0.3878.
The minority class contributes an F1 of exactly 0 (no `struggles`
colony is ever found — recall 0), and macro averaging refuses to let
the majority's easy points hide that.
By hand: the `thrives` one-vs-rest scores are precision
$95/150 = 19/30$, recall $1$, so
$F_1 = \frac{2 \cdot \frac{19}{30} \cdot 1}{\frac{19}{30} + 1}
= \frac{38}{49} \approx 0.7755$;
the `struggles` F1 is $0$; macro-F1
$= \frac{1}{2}\left(0 + \frac{38}{49}\right) = \frac{19}{49} \approx
0.3878$.

### Checkpoint 1

1. Compute, by hand, the accuracy and macro-F1 of the do-nothing rule
   on a validation split with 60 `thrives` and 40 `struggles` rows.
2. Why does the exam's choice of macro-F1 change what a rational
   competitor spends time on, compared with accuracy?
3. From C1: what does the "macro" in macro-F1 signal, and which kind
   of class benefits from it?

## 2. Macro-F1, Derived From Its Pieces

**The derivation, assembled from C1's parts.**
For each class $k$, treat $k$ as positive and everything else as
negative (one-vs-rest), and read three counts off the confusion
matrix $C$ (rows = actual, columns = predicted):

- $TP_k = C_{kk}$ — the diagonal;
- predicted-as-$k$ total $= \sum_i C_{ik}$ — column $k$'s sum;
- actually-$k$ total $= \sum_j C_{kj}$ — row $k$'s sum.

Then, per class (C1's formulas):

$$\text{prec}_k = \frac{C_{kk}}{\text{column-}k\text{ sum}}, \qquad
\text{rec}_k = \frac{C_{kk}}{\text{row-}k\text{ sum}}, \qquad
F_1^{(k)} = \frac{2\,\text{prec}_k\,\text{rec}_k}{\text{prec}_k + \text{rec}_k},$$

$$\text{macro-F1} = \frac{1}{K}\sum_{k} F_1^{(k)}.$$

**Worked example, fully by hand.**
Thirty validation colonies: 10 actually struggle, 20 actually thrive.
A classifier catches 6 of the strugglers (missing 4) and 17 of the
thrivers (missing 3):

$$C = \begin{pmatrix} 6 & 4 \\ 3 & 17 \end{pmatrix}
\quad \text{(rows/cols in the order } \texttt{struggles},
\texttt{thrives}\text{)}$$

`struggles`: column sum $6 + 3 = 9$, row sum $10$, so
$\text{prec} = \frac{6}{9} = \frac{2}{3}$,
$\text{rec} = \frac{6}{10} = \frac{3}{5}$, and
$F_1 = \frac{2 \cdot \frac23 \cdot \frac35}{\frac23 + \frac35}
= \frac{4/5}{19/15} = \frac{12}{19} \approx 0.6316$.

`thrives`: column sum $4 + 17 = 21$, row sum $20$, so
$\text{prec} = \frac{17}{21}$, $\text{rec} = \frac{17}{20}$, and
$F_1 = \frac{2 \cdot \frac{17}{21} \cdot \frac{17}{20}}
{\frac{17}{21} + \frac{17}{20}} = \frac{34}{41} \approx 0.8293$.

$$\text{macro-F1} = \frac{1}{2}\left(\frac{12}{19} + \frac{34}{41}\right)
= \frac{569}{779} \approx 0.7304,$$

while accuracy is $\frac{6 + 17}{30} = \frac{23}{30} \approx 0.7667$ —
higher, because pooling all rows lets the well-served majority
outvote the neglected minority.
That gap is this unit's recurring signature (p11 proves the direction
on a sterner example).

**The computation, once, reusable.**
Sorted label order (`np.unique`) is the convention throughout —
`struggles` is class 0, `thrives` class 1:

In [ ]:
def macro_f1_from_confusion(C):
    # C: (K, K) counts, rows = actual, cols = predicted (sorted label order)
    diag = np.diag(C).astype(float)
    prec = diag / C.sum(axis=0)
    rec = diag / C.sum(axis=1)
    f1 = 2 * prec * rec / (prec + rec)
    return prec, rec, f1, f1.mean()


C_toy = np.array([[6, 4],
                  [3, 17]])
prec, rec, f1, macro = macro_f1_from_confusion(C_toy)
print("prec :", prec.round(4))
print("rec  :", rec.round(4))
print("F1   :", f1.round(4))
print("macro:", round(macro, 4), " (12/19 and 34/41 -> 569/779)")

# sklearn agrees -- rebuild the 30 labels and ask the wrapper:
y_true_toy = np.array(["struggles"] * 10 + ["thrives"] * 20)
y_pred_toy = np.array(["struggles"] * 6 + ["thrives"] * 4
                      + ["struggles"] * 3 + ["thrives"] * 17)
print("sklearn:", round(f1_score(y_true_toy, y_pred_toy, average="macro"), 4))

First principles and wrapper agree to four decimals — the C4 habit:
never trust a wrapper you haven't tested against your own
implementation.
From here on we use `f1_score(..., average="macro")` freely, because
we own its arithmetic.

### Checkpoint 2

1. By hand: per-class F1s and macro-F1 for
   $C = \begin{pmatrix} 8 & 2 \\ 4 & 6 \end{pmatrix}$
   (rows/cols in order `struggles`, `thrives`).
   Compare with the accuracy.
2. In `macro_f1_from_confusion`, which axis sum gives precision's
   denominator and which gives recall's?
   Why (in terms of what each total counts)?
3. When every class has the same F1, how do macro-F1 and that common
   value relate — and does macro-F1 ever exceed the largest per-class
   F1?

## 3. Two Scoreboards on the Real Task

Session 1's minimal submission (scaled 5-NN), judged properly this
time — both metrics, plus the per-class breakdown that explains
them:

In [ ]:
base = Pipeline([("scaler", StandardScaler()),
                 ("knn", KNeighborsClassifier(n_neighbors=5))])
base.fit(X_tr, y_tr)
val_preds = base.predict(X_val)

C_val = confusion_matrix(y_val, val_preds)     # sorted label order
print("confusion (rows=actual, cols=predicted; struggles, thrives):")
print(C_val)

prec, rec, f1, macro = macro_f1_from_confusion(C_val)
acc = (val_preds == y_val).mean()
print()
for k, lab in enumerate(np.unique(y_val)):
    print(f"{lab:>9}: prec {prec[k]:.4f}  rec {rec[k]:.4f}  F1 {f1[k]:.4f}")
print(f"\naccuracy = {acc:.4f} | macro-F1 = {macro:.4f}")
print("sklearn check:", round(f1_score(y_val, val_preds, average='macro'), 4))

The story the breakdown tells:

- `thrives` is comfortable: F1 ≈ 0.84.
- `struggles` lags: recall ≈ 0.67 — a third of the struggling
  colonies are missed, precisely the costly mistake for an apiary.
- Accuracy (0.7867) > macro-F1 (0.7666): the gap *is* the minority
  weakness, surfaced by the metric that refuses to pool it away.

Iteration (Section 5) will be judged by that macro-F1 number and
guided by that breakdown.

### Checkpoint 3

1. From the printed confusion matrix, verify the `struggles` recall
   by hand (which two numbers divide?).
2. A teammate proposes a change that raises `thrives` F1 by 0.01 and
   lowers `struggles` F1 by 0.03.
   What happens to macro-F1, and to accuracy (qualitatively)?
3. Which single confusion-matrix *cell* would you most want to
   shrink, given the apiary's stated purpose?

## 4. Validation Is the Only Honest Signal

**The situation.**
The graded score lives on 200 rows your notebook must never read
(Session 1 §4).
The training table is the entire observable universe, so every honest
estimate has the same shape:

> Carve a seeded, stratified validation split from the training table;
> fit on the rest; measure macro-F1 on the carve.
> **That number — `val_f1` — is the only signal you get**, and every
> modeling decision is a bet placed on it.

Three disciplines keep the signal honest, all C4-grown:

1. **The carve is seeded and frozen** —
   `train_test_split(X, y, test_size=150, random_state=SEED,
   stratify=y)`, pinned once per project.
   Re-rolling the seed until the number improves is C4's Pitfall 4
   ("split shopping") — each re-roll is a peek.
2. **Stratified**, because at 2:1 an unlucky plain split can starve
   the carve of minority rows — the metric's whole focus.
3. **Preprocessing lives inside the pipeline**, so the carve never
   leaks into fitted statistics.

**What validation is not.**
It is not the graded score; it is an *estimate*, and Section 6 shows
the estimate degrades as you lean on it.
The competition mindset: validation is a bank account — every
comparison you run is a withdrawal.

### Checkpoint 4

1. Why stratify the carve *especially* when the grading metric is
   macro-F1?
2. Your friend re-runs the carve with five different seeds and
   reports the best of the five `val_f1`s.
   Name the C4 pitfall and state what the reported number actually
   estimates.
3. In what precise sense is `val_f1` "the only signal"?
   What other numbers exist but are dishonest, and what number is
   honest but unavailable?

## 5. The Iteration Loop

**The loop** — the whole sport in four steps, repeated:

1. **Baseline**: fit the simplest defensible recipe; record
   `val_f1`.
2. **Error analysis**: *look at what the model gets wrong* — the
   confusion breakdown, then the features of the missed rows.
3. **One change**: a single, hypothesis-driven modification.
4. **Re-validate**: same carve, same metric; accept the change only
   if the signal improves.
   Log every attempt — accepted or not.

One change at a time is not bureaucracy: change $k$ and the feature
set together, and whichever result you get, you cannot attribute it.

**Step 1 — baseline** is on the board: scaled 5-NN, `val_f1 = 0.7666`.

**Step 2 — error analysis.**
The breakdown said "minority recall".
Now interrogate the features: which columns actually separate the
classes?
Per-class means on the *training part* (never the carve — error
analysis is a model-fitting activity):

In [ ]:
train_part = pd.DataFrame(X_tr, columns=FEATURES).copy()
train_part["outcome"] = y_tr
class_means = train_part.groupby("outcome").mean().T
class_means["gap_in_stds"] = (
    (class_means["thrives"] - class_means["struggles"]) / X_tr.std()
).abs()
print(class_means.round(2).sort_values("gap_in_stds", ascending=False))

Two clean clusters:

- **Signal columns** — class-mean gaps of 0.5–1.2 standard
  deviations: honey stores, hive mass, varroa index, forager
  traffic, brood frames, temperature swing, queen age.
- **Noise columns** — gaps below 0.2 stds: ambient noise, elevation,
  insulation, hive age, water distance.
  After scaling, each still contributes a full unit of spread to
  every distance — five dimensions of pure static drowning seven of
  signal.

Two hypotheses, then — a $k$ that smooths more, and a feature set
that drops the static.
One at a time.

**Step 3/4 — iteration 1: sweep $k$** (one knob, honest comparison,
logged):

In [ ]:
def val_f1_of(feats, k):
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("knn", KNeighborsClassifier(n_neighbors=k))])
    pipe.fit(X_tr[feats], y_tr)
    return f1_score(y_val, pipe.predict(X_val[feats]), average="macro")


log = [{"step": "baseline", "change": "scaled 5-NN, all features",
        "val_f1": val_f1_of(FEATURES, 5)}]

ks = np.array([1, 3, 5, 7, 9, 11])
sweep = np.array([val_f1_of(FEATURES, k) for k in ks])
for k, v in zip(ks, sweep):
    print(f"k={k:2d}: val macro-F1 = {v:.4f}")

best_k = int(ks[np.argmax(sweep)])       # argmax = first max = smallest tied k
log.append({"step": "iter-1", "change": f"k swept over {[int(k) for k in ks]} -> {best_k}",
            "val_f1": sweep.max()})
print("\naccepted: k =", best_k)

In [ ]:
# iteration 2: drop the noise columns (k stays at the accepted 11)
SIGNAL = ["honey_stores_kg", "autumn_hive_mass_kg", "varroa_mite_index",
          "forager_traffic_per_min", "brood_frames", "daily_temp_swing_c",
          "queen_age_years"]

f1_signal = val_f1_of(SIGNAL, best_k)
log.append({"step": "iter-2", "change": "12 features -> 7 SIGNAL columns",
            "val_f1": f1_signal})

log_df = pd.DataFrame(log)
print(log_df.to_string(index=False,
                       formatters={"val_f1": "{:.4f}".format}))

The log reads: 0.7666 → 0.8103 (smoothing helped) → 0.8199 (static
removed).
Each row is one hypothesis, one change, one honest re-measurement —
a grader (or teammate, or you-in-a-week) can replay the entire
campaign from the log alone.
Session 3's writeup rubric will demand exactly this trail.

### Checkpoint 5

1. Why must error analysis (the per-class means table) use only the
   training part?
   What quiet leak happens if it uses the whole table?
2. Iteration 2 was tested *at the accepted* $k = 11$ only.
   What is the argument for that choice, and what is the argument for
   re-sweeping $k$ on the SIGNAL columns?
   (Both are defensible — name the cost the second incurs.)
3. Your iteration 3 idea lowers `val_f1` from 0.8199 to 0.8050.
   What goes in the log, and what does the final model use?

## 6. Wearing Out the Validation Split

**The stated fact** (this course's register — the full expectation
argument is p12's subject):

> Every time you compare candidates on the validation split and keep
> the winner, the winning `val_f1` becomes a slightly **optimistic**
> estimate of held-back performance.
> The more candidates compared, the larger the optimism — even when
> none of the candidates is genuinely better than another.

The mechanism in one sentence: the maximum of several noisy numbers
is biased upward — selection converts noise into apparent skill.
You have seen the ingredient before (C4's Pitfall: "reporting the
best fold"); here it applies to *your own iteration loop*.

How big is the noise being selected over?
Re-run one fixed recipe (scaled 5-NN) under eight different carve
seeds — no model change at all:

In [ ]:
spread = []
for rs in range(8):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=150,
                                      random_state=rs, stratify=y)
    p = Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=5))]).fit(Xa, ya)
    spread.append(f1_score(yb, p.predict(Xb), average="macro"))
spread = np.array(spread)
print("same recipe, eight carves:", spread.round(4))
print("spread:", round(spread.max() - spread.min(), 4))

A spread of several hundredths *from split luck alone* — the same
order as our iteration gains.
Conclusions, in competition-craft form:

- **Cap the loop.** A handful of deliberate, hypothesis-driven
  iterations (this unit's exercises cap at three) beats dozens of
  micro-tweaks — the dozens select on noise.
- **Freeze the carve.** Comparing candidates across *different*
  carves adds carve-luck to candidate-luck.
- **Trust differences, not digits.** A gain of 0.002 on 150
  validation rows is weather; a gain comparable to the whole
  seed-luck spread, backed by a mechanism you can explain (smoothing,
  static removal), is signal.
- **Expect the graded score to land below your last `val_f1`.**
  You optimized against this split; the held-back rows never agreed
  to that.

### Checkpoint 6

1. State the stated fact from memory, and name the mechanism in
   three words or fewer.
2. Given the printed spread, would you accept an iteration that
   improves `val_f1` by 0.003?
   By 0.06?
   Justify with the weather/signal rule.
3. Why does "freeze the carve" and "cap the loop" *together* imply
   keeping a log?
   (What question can only the log answer at writeup time?)

## 7. Worked Exam-Style Example: Choosing k by the Right Metric

> **Task.**
> On the apiary table, with the pinned carve
> (`test_size=150, random_state=SEED, stratify=y`):
> **(a)** for each `k` in `ks = [1, 3, 5, 7, 9, 11]`, fit a
> scaler+kNN `Pipeline` on the training part and record `val_f1s[i]`,
> the validation **macro-F1** (shape `(6,)`);
> **(b)** set `best_k` to the `k` with the highest `val_f1s`,
> **smallest `k` winning ties** (an `int`);
> **(c)** report `gap` — the (accuracy − macro-F1) difference on the
> validation split for the `best_k` model (a `float`).
> **Constraints (zero points): any supervised estimator other than
> `KNeighborsClassifier` (the usual families and hand
> re-implementations included); choosing `k` by accuracy; touching
> the held-back split.**

**Solution, narrated.**
(a) is Section 5's sweep verbatim; (b) is the C4 tie idiom —
`np.argmax` returns the first maximum, and `ks` ascends; (c) makes
you state the imbalance gap as a number, the register's way of
checking you know *why* the metric was chosen.

In [ ]:
val_f1s = np.array([val_f1_of(FEATURES, int(k)) for k in ks])
print("ks     :", ks)
print("val_f1s:", val_f1s.round(4))

best_k = int(ks[np.argmax(val_f1s)])
print("best_k :", best_k)

final = Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=best_k))])
final.fit(X_tr, y_tr)
preds = final.predict(X_val)
gap = (preds == y_val).mean() - f1_score(y_val, preds, average="macro")
print("gap    :", round(gap, 4))

Grader's-eye view: deliverables `val_f1s`, `best_k`, `gap` under
exactly those names; the instant-zero mistakes are sweeping with
`.score(...)` (that's accuracy — the banned scoreboard) or any read
of the held-back rows.
The gap (~0.016 here) is small because $k = 11$ already serves the
minority decently — for the *baseline* it was 0.02, and for the
do-nothing rule it was 0.25.

### Checkpoint 7

1. Why does the constraint ban choosing `k` by accuracy when the
   graded metric is macro-F1?
   Construct (in words) a case where the two sweeps pick different
   `k`.
2. Which line of the solution implements the tie rule, and why does
   it need no extra code?
3. If `val_f1s` came out `[0.71, 0.79, 0.79, 0.78, 0.79, 0.77]`,
   what is `best_k`?

## 8. Common Pitfalls II

**Pitfall 1 — measuring the metric on the training part.**
`f1_score(y_tr, pipe.predict(X_tr), ...)` is C1's memorization
scoreboard: for small $k$ it reports near-perfection and estimates
nothing.
Fix: metrics only on rows the fit never saw.

**Pitfall 2 — optimizing one metric, reporting another.**
Sweeping $k$ by accuracy and then reporting the winner's macro-F1
quietly optimizes the wrong target; under imbalance the
accuracy-best $k$ can differ from the macro-F1-best $k$.
Fix: the sweep's judge and the reported metric are the same function,
pinned at the top of the notebook.

**Pitfall 3 — an unstratified carve under imbalance.**
A plain 150-row carve can draw 40-something `struggles` rows instead
of 55; minority F1 then rides on a handful of rows and the whole
signal gets noisier.
Fix: `stratify=y`, always, and check the carve's class counts once
(Section 1's printout).

**Pitfall 4 — iterating on the carve until it agrees with you.**
Twenty tweaks at +0.003 each is not a +0.06 improvement; it is
selection on noise (Section 6), and the graded score will say so.
Fix: cap, log, and demand a mechanism for every accepted change.

### Checkpoint 8

1. A notebook prints `train macro-F1 = 0.997, val macro-F1 = 0.78`
   for $k = 1$.
   Which pitfall, which C1 concept, and which number (if either) is
   reportable?
2. Explain how Pitfall 2 could make a submission *look* honest (all
   numbers computed on validation!) while still being
   metric-confused.
3. Which pitfall is the *hardest to detect from the notebook alone*,
   and what Session-3 artifact makes it visible?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Accuracy $= 60/100 = 0.6$.
   `thrives`: prec $= 60/100 = 0.6$, rec $= 1$,
   $F_1 = 2(0.6)/1.6 = 0.75$; `struggles`: $F_1 = 0$;
   macro-F1 $= 0.375$.
2. Under accuracy, effort flows to the 63% majority (each majority
   row is worth as much as each minority row and there are more of
   them); under macro-F1, half the score rides on the 37% minority,
   so minority recall becomes the profitable place to work.
3. Every class counts equally in the average — rare classes gain the
   most, because pooled metrics let frequent classes outvote them.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `struggles`: prec $= 8/12 = 2/3$, rec $= 8/10 = 4/5$,
   $F_1 = \frac{2 \cdot \frac23 \cdot \frac45}{\frac23 + \frac45}
   = \frac{16/15}{22/15} = \frac{8}{11} \approx 0.727$.
   `thrives`: prec $= 6/8 = 3/4$, rec $= 6/10 = 3/5$,
   $F_1 = \frac{2 \cdot \frac34 \cdot \frac35}{\frac34 + \frac35}
   = \frac{9/10}{27/20} = \frac{2}{3} \approx 0.667$.
   Macro-F1: $\frac{8}{11} = \frac{24}{33}$ and
   $\frac{2}{3} = \frac{22}{33}$, so the mean is
   $\frac{23}{33} \approx 0.697$.
   Accuracy $= 14/20 = 0.7$ — nearly equal to macro-F1 here, because
   both classes are served comparably.
2. `C.sum(axis=0)` (column sums — everything *predicted* class $k$)
   for precision; `C.sum(axis=1)` (row sums — everything *actually*
   class $k$) for recall.
3. Macro-F1 equals the common value (the mean of equal numbers); and
   no — a mean can never exceed its largest term.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Recall of `struggles` $= 37/(37+18) = 37/55 \approx 0.673$ — the
   diagonal cell over its row sum.
2. Macro-F1 falls (the average moves by $(0.01 - 0.03)/2 = -0.01$);
   accuracy likely *rises*, since `thrives` rows are numerous — a
   clean example of the two scoreboards disagreeing about the same
   change.
3. $C_{01} = 18$: actually-struggling colonies predicted `thrives` —
   the missed interventions the task exists to prevent.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Macro-F1 gives half its weight to the minority class, so the
   minority F1 must be *measurable* — stratification guarantees the
   carve holds its ~55 minority rows rather than a lucky-draw count.
2. Split shopping (C4 Pitfall 4).
   The best-of-five is the maximum of five noisy estimates of the
   same quantity — an upward-biased order statistic, not an estimate
   of the recipe's quality.
3. Honest and available: `val_f1` on the frozen carve.
   Available but dishonest: training-set metrics (memorization),
   best-of-several-seeds (selection).
   Honest but unavailable: the held-back macro-F1 — the protocol
   forbids computing it.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Error analysis chooses features — a modeling decision.
   Done on the full table, the carve's rows help choose what the
   model attends to, and the carve stops being unseen (the same
   logic as fitting the scaler on all rows, one level up).
2. For: one change at a time — re-sweeping $k$ *and* changing
   features simultaneously confounds attribution.
   Against: the best $k$ for 7 columns may differ from 12 columns'.
   The cost: a re-sweep is five more validation comparisons —
   another withdrawal from the account (Section 6).
3. The attempt goes in the log with its 0.8050 — rejected changes
   are recorded, not erased.
   The final model uses the last *accepted* state: SIGNAL columns,
   $k = 11$, `val_f1 = 0.8199`.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Selecting the best of several validation-compared candidates makes
   the winner's `val_f1` optimistic, increasingly so with more
   candidates — mechanism: "max of noise" (selection bias).
2. 0.003 is far inside the ~0.057 seed-luck spread: weather —
   reject (or re-confirm under a sterner protocol).
   0.06 exceeds the whole spread and (with a mechanism) is signal —
   accept.
3. The cap is only auditable if attempts are counted, and the
   optimism warning is only calibratable if you know *how many*
   comparisons the final number survived — the log answers "how many
   candidates did this `val_f1` beat?", which the writeup must
   disclose.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. The graded metric weighs the minority at 50%; accuracy weighs it
   at ~37%.
   A large-$k$ model that absorbs the minority into the majority can
   win accuracy (many easy majority rows correct) while a smaller-$k$
   model with sharper minority recall wins macro-F1 — the sweeps
   then disagree.
2. `best_k = int(ks[np.argmax(val_f1s)])` — `np.argmax` returns the
   *first* maximum and `ks` is ascending, so the smallest tied `k`
   wins for free.
3. `best_k = 3`: the maximum 0.79 first occurs at index 1.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1; C1's overfitting/memorization ($k = 1$ stores the
   training set, so training metrics are ~perfect by construction).
   Only the 0.78 validation number is reportable.
2. Every printed number *is* validation-computed — but the sweep's
   `argmax` ran over validation *accuracy*, so the model was chosen
   by the wrong judge; the reported macro-F1 is honest arithmetic
   about a dishonestly-chosen model.
3. Pitfall 4 — the notebook shows only the surviving model, not the
   forty quiet comparisons behind it.
   Session 3's iteration log (a required writeup artifact) is what
   makes the loop's length visible.

</details>